In [ ]:
# RAINFALL PATTERN ANALYSIS
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# ============================================================================
# PART 1: GENERATE MESSY DATASET (2500 ROWS)
# ============================================================================

print("="*60)
print("STEP 1: Generating 2500 rows of messy rainfall data...")
print("="*60)

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Configuration
num_rows = 2500
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Pune']
states = {'Mumbai': 'MH', 'Delhi': 'DL', 'Bangalore': 'KA', 'Chennai': 'TN', 'Pune': 'MH'}
rainfall_types = ['Light Rain', 'Moderate', 'Heavy Rain', 'Drizzle', 'None', 'light rain', 'LIGHT RAIN', 'Moderate Rain']
seasons = ['Winter', 'Summer', 'Autumn', 'Monsoon']

# Generate dates
start_date = datetime(2020, 1, 1)
dates = [start_date + timedelta(days=i) for i in range(num_rows)]

# Create messy dataset
data = []
for i, date in enumerate(dates):
    city = random.choice(cities)
    state = states[city]

    # Intentional Mess: Mixed Date Formats
    if i % 10 == 0:
        date_str = date.strftime('%d-%m-%Y')
    elif i % 7 == 0:
        date_str = date.strftime('%d/%m/%Y')
    elif i % 13 == 0:
        date_str = date.strftime('%d-%m-%y')
    else:
        date_str = date.strftime('%d/%m/%Y')

    # Rainfall: 10% missing values with different representations
    rainfall = np.random.randint(0, 200)
    if random.random() < 0.10:
        rainfall = random.choice([np.nan, '', 'NA', None, ''])

    # Temperature: 8% missing values
    temp = round(np.random.normal(30, 5), 1)
    if random.random() < 0.08:
        temp = random.choice([np.nan, '', 'NA', None])

    # Humidity: 7% missing values
    humidity = np.random.randint(40, 100)
    if random.random() < 0.07:
        humidity = random.choice([np.nan, '', 'NA', None])

    # Season: 5% missing
    season = random.choice(seasons)
    if random.random() < 0.05:
        season = random.choice(['', np.nan, None])

    # Rainfall Type: 12% messy/inconsistent
    rainfall_type = random.choice(rainfall_types)
    if random.random() < 0.12:
        rainfall_type = random.choice(['', 'None', np.nan, 'Moderate Rain', 'light rain', 'LIGHT RAIN'])

    row = [date_str, city, state, rainfall, temp, humidity, season, rainfall_type]
    data.append(row)

# Create DataFrame
columns = ['Date', 'City', 'State', 'Rainfall_mm', 'Temperature_C', 'Humidity_%', 'Season', 'Rainfall_Type']
df_raw = pd.DataFrame(data, columns=columns)

# Save the messy dataset
df_raw.to_csv('rainfall_uncleaned_2500.csv', index=False)
print(f"✅ Generated {len(df_raw)} rows of messy data")
print(f"📁 Saved as 'rainfall_uncleaned_2500.csv'")
print("\nFirst 5 rows of messy data:")
print(df_raw.head())

STEP 1: Generating 2500 rows of messy rainfall data...
✅ Generated 2500 rows of messy data
📁 Saved as 'rainfall_uncleaned_2500.csv'

First 5 rows of messy data:
         Date    City State Rainfall_mm Temperature_C Humidity_%   Season  \
0  01-01-2020  Mumbai    MH          NA          27.2         47   Winter   
1  02/01/2020  Mumbai    MH                      32.6         60   Summer   
2  03/01/2020    Pune    MH         102          25.4         50   Summer   
3  04/01/2020   Delhi    DL          87          29.4         92   Winter   
4  05/01/2020  Mumbai    MH          99          32.0         92  Monsoon   

  Rainfall_Type  
0      Moderate  
1    LIGHT RAIN  
2    light rain  
3    light rain  
4          None  


In [ ]:
# ============================================================================
# PART 2: DATA CLEANING
# ============================================================================

print("\n" + "="*60)
print("STEP 2: Data Cleaning")
print("="*60)

# Create a copy for cleaning
df = df_raw.copy()

# 2.1: Standardize Date Format
print("\n📅 Standardizing Date Formats...")
def clean_date(date_str):
    try:
        # Try multiple date formats
        for fmt in ['%d-%m-%Y', '%d/%m/%Y', '%d-%m-%y']:
            try:
                return pd.to_datetime(date_str, format=fmt)
            except:
                continue
        # If all fail, try pandas auto-detection
        return pd.to_datetime(date_str)
    except:
        return pd.NaT

df['Clean_Date'] = df['Date'].apply(clean_date)
df['Year'] = df['Clean_Date'].dt.year
df['Month'] = df['Clean_Date'].dt.month
df['Month_Name'] = df['Clean_Date'].dt.strftime('%B')

# 2.2: Handle Missing Values
print("\n🧹 Handling Missing Values...")
print(f"Missing values before cleaning:")
print(df[['Rainfall_mm', 'Temperature_C', 'Humidity_%']].isna().sum())

# Convert 'Rainfall_mm', 'Temperature_C', 'Humidity_%' to numeric first
df['Rainfall_mm'] = pd.to_numeric(df['Rainfall_mm'], errors='coerce')
df['Temperature_C'] = pd.to_numeric(df['Temperature_C'], errors='coerce')
df['Humidity_%'] = pd.to_numeric(df['Humidity_%'], errors='coerce')

# Remove rows where Rainfall_mm is missing (critical column)
df = df.dropna(subset=['Rainfall_mm'])

# For Temperature and Humidity, fill with mean by city
df['Temperature_C'] = df.groupby('City')['Temperature_C'].transform(
    lambda x: x.fillna(pd.to_numeric(x, errors='coerce').mean())
)
df['Humidity_%'] = df.groupby('City')['Humidity_%'].transform(
    lambda x: x.fillna(pd.to_numeric(x, errors='coerce').mean())
)

# 2.3: Convert Text to Numbers (This step is now mostly redundant for these columns, but kept for other potential text-to-number conversions if any)
# The conversions for Rainfall_mm, Temperature_C, Humidity_% are now done earlier.

# 2.4: Clean Inconsistent Categories
print("\n🔄 Cleaning Text Categories...")
def clean_rainfall_type(text):
    if pd.isna(text) or text == '' or text == 'None':
        return 'No Rain'
    text = str(text).strip().title()
    # Map variations to standard categories
    mapping = {
        'Light Rain': 'Light Rain',
        'Light': 'Light Rain',
        'Moderate': 'Moderate Rain',
        'Moderate Rain': 'Moderate Rain',
        'Heavy': 'Heavy Rain',
        'Heavy Rain': 'Heavy Rain',
        'Drizzle': 'Drizzle',
        'None': 'No Rain',
        'No Rain': 'No Rain'
    }
    return mapping.get(text, 'Other')

df['Clean_Rainfall_Type'] = df['Rainfall_Type'].apply(clean_rainfall_type)

# 2.5: Handle Outliers
print("\n📊 Handling Outliers...")
Q1 = df['Rainfall_mm'].quantile(0.25)
Q3 = df['Rainfall_mm'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['Is_Outlier'] = (df['Rainfall_mm'] < lower_bound) | (df['Rainfall_mm'] > upper_bound)
print(f"Outliers detected: {df['Is_Outlier'].sum()} rows")

# Keep outliers but flag them (we'll keep them for analysis)

# 2.6: Create Rainfall Category
print("\n🏷️ Creating Rainfall Categories...")
def categorize_rainfall(rainfall):
    if rainfall < 10:
        return 'Light'
    elif rainfall < 50:
        return 'Moderate'
    elif rainfall < 100:
        return 'Heavy'
    else:
        return 'Extreme'

df['Rainfall_Category'] = df['Rainfall_mm'].apply(categorize_rainfall)

# Final cleaned dataset
print(f"\n✅ Cleaning complete! Final dataset shape: {df.shape}")
print("\nFirst 5 rows of cleaned data:")
print(df.head())

# Save cleaned dataset
df.to_csv('rainfall_cleaned.csv', index=False)


STEP 2: Data Cleaning

📅 Standardizing Date Formats...

🧹 Handling Missing Values...
Missing values before cleaning:
Rainfall_mm      104
Temperature_C    104
Humidity_%        85
dtype: int64

🔄 Cleaning Text Categories...

📊 Handling Outliers...
Outliers detected: 0 rows

🏷️ Creating Rainfall Categories...

✅ Cleaning complete! Final dataset shape: (2245, 15)

First 5 rows of cleaned data:
         Date    City State  Rainfall_mm  Temperature_C  Humidity_%   Season  \
2  03/01/2020    Pune    MH        102.0           25.4        50.0   Summer   
3  04/01/2020   Delhi    DL         87.0           29.4        92.0   Winter   
4  05/01/2020  Mumbai    MH         99.0           32.0        92.0  Monsoon   
5  06/01/2020    Pune    MH          1.0           25.4        63.0            
6  07/01/2020   Delhi    DL        157.0           27.1        60.0   Autumn   

  Rainfall_Type Clean_Date  Year  Month Month_Name Clean_Rainfall_Type  \
2    light rain 2020-01-03  2020      1    Januar

In [ ]:
# ============================================================================
# PART 3: ANALYSIS
# ============================================================================

print("\n" + "="*60)
print("STEP 3: Data Analysis")
print("="*60)

# 3.1: Basic Statistics
print("\n📈 Basic Statistics:")
print(df[['Rainfall_mm', 'Temperature_C', 'Humidity_%']].describe())

# 3.2: Monthly Rainfall Trends
monthly_rainfall = df.groupby('Month')['Rainfall_mm'].agg(['mean', 'sum', 'count']).reset_index()
monthly_rainfall.columns = ['Month', 'Avg_Rainfall', 'Total_Rainfall', 'Rainy_Days']
print("\n📊 Monthly Rainfall Summary:")
print(monthly_rainfall)

# 3.3: Seasonal Comparison
seasonal_rainfall = df.groupby('Season')['Rainfall_mm'].agg(['mean', 'sum', 'count']).reset_index()
seasonal_rainfall.columns = ['Season', 'Avg_Rainfall', 'Total_Rainfall', 'Rainy_Days']
print("\n🌦️ Seasonal Rainfall Summary:")
print(seasonal_rainfall)

# 3.4: City-wise Rainfall Analysis
city_rainfall = df.groupby('City')['Rainfall_mm'].agg(['mean', 'sum', 'max', 'min', 'count']).reset_index()
city_rainfall.columns = ['City', 'Avg_Rainfall', 'Total_Rainfall', 'Max_Rainfall', 'Min_Rainfall', 'Rainy_Days']
print("\n🏙️ City-wise Rainfall Summary:")
print(city_rainfall)

# 3.5: Correlation Analysis
correlation_rainfall_temp = df['Rainfall_mm'].corr(df['Temperature_C'])
correlation_rainfall_humidity = df['Rainfall_mm'].corr(df['Humidity_%'])
correlation_temp_humidity = df['Temperature_C'].corr(df['Humidity_%'])

print("\n🔗 Correlation Analysis:")
print(f"Correlation (Rainfall vs Temperature): {correlation_rainfall_temp:.3f}")
print(f"Correlation (Rainfall vs Humidity): {correlation_rainfall_humidity:.3f}")
print(f"Correlation (Temperature vs Humidity): {correlation_temp_humidity:.3f}")

# 3.6: Rainfall Type Distribution
rainfall_type_counts = df['Clean_Rainfall_Type'].value_counts()
print("\n🌧️ Rainfall Type Distribution:")
print(rainfall_type_counts)



STEP 3: Data Analysis

📈 Basic Statistics:
       Rainfall_mm  Temperature_C   Humidity_%
count   2245.00000    2245.000000  2245.000000
mean     100.75902      30.196324    69.892421
std       58.09667       4.933665    16.795228
min        0.00000      15.700000    40.000000
25%       50.00000      27.000000    56.000000
50%      102.00000      30.291451    69.530151
75%      152.00000      33.200000    84.000000
max      199.00000      49.600000    99.000000

📊 Monthly Rainfall Summary:
    Month  Avg_Rainfall  Total_Rainfall  Rainy_Days
0       1     99.086735         19421.0         196
1       2     97.486339         17840.0         183
2       3     95.304124         18489.0         194
3       4    101.458763         19683.0         194
4       5    102.870270         19031.0         185
5       6     98.810056         17687.0         179
6       7     97.310881         18781.0         193
7       8    101.126263         20023.0         198
8       9    102.633508         1960

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# ============================================================================
# VISUALIZATION 1: Interactive Monthly Rainfall Trend with Hover
# ============================================================================

print("\n📈 Creating Visualization 1: Monthly Rainfall Trend")

monthly_stats = df.groupby('Month_Name')['Rainfall_mm'].agg(['mean', 'std', 'count']).reset_index()
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_stats['Month_Order'] = monthly_stats['Month_Name'].map({month: i for i, month in enumerate(month_order)})
monthly_stats = monthly_stats.sort_values('Month_Order')
monthly_stats['se'] = monthly_stats['std'] / np.sqrt(monthly_stats['count'])

fig1 = go.Figure()

# Add line with markers
fig1.add_trace(go.Scatter(
    x=monthly_stats['Month_Name'],
    y=monthly_stats['mean'],
    mode='lines+markers+text',
    name='Average Rainfall',
    line=dict(color='#2E86AB', width=3),
    marker=dict(size=12, color='#2E86AB', symbol='circle'),
    text=monthly_stats['mean'].round(1),
    textposition='top center',
    hovertemplate='<b>Month:</b> %{x}<br>' +
                  '<b>Average Rainfall:</b> %{y:.1f} mm<br>' +
                  '<b>Standard Deviation:</b> %{text:.1f} mm<br>' +
                  '<b>Sample Size:</b> %{customdata[0]} days<extra></extra>',
    customdata=monthly_stats[['count']].values
))

# Add confidence interval
fig1.add_trace(go.Scatter(
    x=monthly_stats['Month_Name'].tolist() + monthly_stats['Month_Name'].tolist()[::-1],
    y=(monthly_stats['mean'] + 1.96 * monthly_stats['se']).tolist() +
       (monthly_stats['mean'] - 1.96 * monthly_stats['se']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(46, 134, 171, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Confidence Interval',
    hoverinfo='skip'
))

fig1.update_layout(
    title='📊 Monthly Rainfall Trend with Confidence Interval',
    xaxis_title='Month',
    yaxis_title='Average Rainfall (mm)',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    width=800,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig1.show()






📈 Creating Visualization 1: Monthly Rainfall Trend


In [ ]:
# ============================================================================
# VISUALIZATION 2: Interactive City-wise Rainfall Box Plot with Hover
# ============================================================================

print("📊 Creating Visualization 2: City-wise Rainfall Distribution")

fig2 = go.Figure()

city_order = df.groupby('City')['Rainfall_mm'].median().sort_values(ascending=False).index

for city in city_order:
    city_data = df[df['City'] == city]['Rainfall_mm']
    fig2.add_trace(go.Box(
        y=city_data,
        name=city,
        boxmean='sd',
        hovertemplate='<b>City:</b> %{x}<br>' +
                      '<b>Median:</b> %{median:.1f} mm<br>' +
                      '<b>Mean:</b> %{mean:.1f} mm<br>' +
                      '<b>Std Dev:</b> %{sd:.1f} mm<br>' +
                      '<b>Min:</b> %{min:.1f} mm<br>' +
                      '<b>Max:</b> %{max:.1f} mm<br>' +
                      '<b>Q1:</b> %{q1:.1f} mm<br>' +
                      '<b>Q3:</b> %{q3:.1f} mm<extra></extra>'
    ))

fig2.update_layout(
    title='🏙️ Rainfall Distribution by City',
    xaxis_title='City',
    yaxis_title='Rainfall (mm)',
    template='plotly_white',
    height=500,
    width=800,
    showlegend=False
)

fig2.show()


📊 Creating Visualization 2: City-wise Rainfall Distribution


In [ ]:

# ============================================================================
# VISUALIZATION 3: Interactive Seasonal Rainfall Comparison with Hover
# ============================================================================

print("🌦️ Creating Visualization 3: Seasonal Rainfall Comparison")

seasonal_stats = df.groupby('Season')['Rainfall_mm'].agg(['mean', 'std', 'count']).reset_index()
season_order = ['Winter', 'Summer', 'Autumn', 'Monsoon']
seasonal_stats['Season'] = pd.Categorical(seasonal_stats['Season'], categories=season_order, ordered=True)
seasonal_stats = seasonal_stats.sort_values('Season')

# Filter out NaN seasons and reset index before plotting
seasonal_stats = seasonal_stats.dropna(subset=['Season']).reset_index(drop=True)

colors = ['#4A90D9', '#F5A623', '#7ED321', '#D0021B']

fig3 = go.Figure()

for i, row in seasonal_stats.iterrows():
    fig3.add_trace(go.Bar(
        x=[row['Season']],
        y=[row['mean']],
        name=row['Season'],
        marker_color=colors[i],
        error_y=dict(
            type='data',
            array=[row['std']],
            visible=True,
            color='rgba(0,0,0,0.5)',
            thickness=2,
            width=10
        ),
        hovertemplate='<b>Season:</b> %{x}<br>' +
                      '<b>Average Rainfall:</b> %{y:.1f} mm<br>' +
                      '<b>Standard Deviation:</b> %{customdata[0]:.1f} mm<br>' +
                      '<b>Sample Size:</b> %{customdata[1]} days<extra></extra>',
        customdata=[[row['std'], row['count']]]
    ))

fig3.update_layout(
    title='🌧️ Average Rainfall by Season',
    xaxis_title='Season',
    yaxis_title='Average Rainfall (mm)',
    template='plotly_white',
    height=500,
    width=800,
    showlegend=False
)

fig3.show()


🌦️ Creating Visualization 3: Seasonal Rainfall Comparison


In [ ]:

# ============================================================================
# VISUALIZATION 4: Interactive Rainfall Heatmap with Hover
# ============================================================================

print("🗺️ Creating Visualization 4: Rainfall Heatmap")

pivot_heatmap = df.pivot_table(values='Rainfall_mm', index='City', columns='Month_Name', aggfunc='mean')
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
pivot_heatmap = pivot_heatmap[month_order]

fig4 = go.Figure(data=go.Heatmap(
    z=pivot_heatmap.values,
    x=pivot_heatmap.columns,
    y=pivot_heatmap.index,
    colorscale='YlOrRd',
    text=pivot_heatmap.values.round(1),
    texttemplate='%{text}',
    textfont={"size": 10},
    hovertemplate='<b>City:</b> %{y}<br>' +
                  '<b>Month:</b> %{x}<br>' +
                  '<b>Rainfall:</b> %{z:.1f} mm<extra></extra>'
))

fig4.update_layout(
    title='🗺️ Monthly Rainfall Heatmap by City',
    xaxis_title='Month',
    yaxis_title='City',
    template='plotly_white',
    height=500,
    width=800
)

fig4.show()

🗺️ Creating Visualization 4: Rainfall Heatmap


In [ ]:
# ============================================================================
# VISUALIZATION 5: Interactive Correlation Matrix with Hover
# ============================================================================

print("🔗 Creating Visualization 5: Correlation Matrix")

correlation_data = df[['Rainfall_mm', 'Temperature_C', 'Humidity_%']].corr()
correlation_data.index = ['Rainfall', 'Temperature', 'Humidity']
correlation_data.columns = ['Rainfall', 'Temperature', 'Humidity']

fig5 = go.Figure(data=go.Heatmap(
    z=correlation_data.values,
    x=correlation_data.columns,
    y=correlation_data.index,
    colorscale='RdBu_r',
    zmid=0,
    text=correlation_data.values.round(3),
    texttemplate='%{text}',
    textfont={"size": 12},
    hovertemplate='<b>Variables:</b> %{y} vs %{x}<br>' +
                  '<b>Correlation:</b> %{z:.3f}<extra></extra>'
))

fig5.update_layout(
    title='🔗 Correlation Matrix (Rainfall, Temperature, Humidity)',
    template='plotly_white',
    height=500,
    width=600
)

fig5.show()


🔗 Creating Visualization 5: Correlation Matrix


In [ ]:
# ============================================================================
# VISUALIZATION 6: Interactive Scatter Plot with Hover
# ============================================================================

print("📉 Creating Visualization 6: Rainfall vs Humidity Scatter Plot")

# Sample for better performance
df_sample = df.sample(min(1000, len(df)))

fig6 = go.Figure()

# Add scatter points
fig6.add_trace(go.Scatter(
    x=df_sample['Humidity_%'],
    y=df_sample['Rainfall_mm'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_sample['Temperature_C'],
        colorscale='RdBu',
        showscale=True,
        colorbar=dict(title='Temperature (°C)')
,        opacity=0.6,
        line=dict(width=0.5, color='white')
    ),
    text=df_sample['City'] + '<br>Season: ' + df_sample['Season'],
    hovertemplate='<b>City:</b> %{text}<br>' +
                  '<b>Rainfall:</b> %{y:.1f} mm<br>' +
                  '<b>Humidity:</b> %{x:.0f}%<br>' +
                  '<b>Temperature:</b> %{marker.color:.1f}°C<br>' +
                  '<b>Date:</b> %{customdata}<extra></extra>',
    customdata=df_sample['Clean_Date'].dt.strftime('%Y-%m-%d')
))

# Add trendline
z = np.polyfit(df_sample['Humidity_%'], df_sample['Rainfall_mm'], 1)
p = np.poly1d(z)
x_trend = np.linspace(df_sample['Humidity_%'].min(), df_sample['Humidity_%'].max(), 100)

fig6.add_trace(go.Scatter(
    x=x_trend,
    y=p(x_trend),
    mode='lines',
    name=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}',
    line=dict(color='red', width=2, dash='dash'),
    hovertemplate='<b>Trend Line</b><br>%{y:.1f} mm<extra></extra>'
))

fig6.update_layout(
    title='💧 Rainfall vs Humidity with Trendline',
    xaxis_title='Humidity (%)',
    yaxis_title='Rainfall (mm)',
    template='plotly_white',
    height=500,
    width=800,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig6.show()

📉 Creating Visualization 6: Rainfall vs Humidity Scatter Plot


In [ ]:
# ============================================================================
# VISUALIZATION 7: Interactive Donut Chart with Hover
# ============================================================================

print("🍩 Creating Visualization 7: Rainfall Category Distribution")

category_counts = df['Rainfall_Category'].value_counts().reset_index()
category_counts.columns = ['Category', 'Count']

colors = ['#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF']

fig7 = go.Figure(data=go.Pie(
    labels=category_counts['Category'],
    values=category_counts['Count'],
    hole=0.4,
    marker=dict(colors=colors),
    textinfo='label+percent',
    textposition='outside',
    hovertemplate='<b>Category:</b> %{label}<br>' +
                  '<b>Count:</b> %{value:,}<br>' +
                  '<b>Percentage:</b> %{percent}<br>' +
                  '<extra></extra>',
    pull=[0.05, 0, 0, 0],
    textfont=dict(size=12)
))

fig7.update_layout(
    title='🍩 Rainfall Category Distribution',
    template='plotly_white',
    height=500,
    width=600,
    annotations=[dict(text='Rainfall<br>Categories', x=0.5, y=0.5, font_size=16, showarrow=False)]
)

fig7.show()


🍩 Creating Visualization 7: Rainfall Category Distribution


In [ ]:
# ============================================================================
# VISUALIZATION 8: Interactive Pie Chart with Hover
# ============================================================================

print("📊 Creating Visualization 8: Rainfall Type Distribution")

type_counts = df['Clean_Rainfall_Type'].value_counts().reset_index()
type_counts.columns = ['Type', 'Count']

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

fig8 = go.Figure(data=go.Pie(
    labels=type_counts['Type'],
    values=type_counts['Count'],
    marker=dict(colors=colors),
    textinfo='label+percent',
    textposition='outside',
    hovertemplate='<b>Rainfall Type:</b> %{label}<br>' +
                  '<b>Count:</b> %{value:,}<br>' +
                  '<b>Percentage:</b> %{percent}<br>' +
                  '<extra></extra>',
    pull=[0.05, 0, 0, 0, 0],
    textfont=dict(size=11)
))

fig8.update_layout(
    title='🌧️ Rainfall Type Distribution',
    template='plotly_white',
    height=500,
    width=600
)

fig8.show()

📊 Creating Visualization 8: Rainfall Type Distribution


In [ ]:

# ============================================================================
# VISUALIZATION 9: Interactive Temperature Distribution with Hover
# ============================================================================

print("🌡️ Creating Visualization 9: Temperature Distribution")

fig9 = go.Figure()

# Histogram
fig9.add_trace(go.Histogram(
    x=df['Temperature_C'],
    nbinsx=30,
    name='Frequency',
    marker=dict(color='#FF6B6B', line=dict(color='black', width=0.5)),
    hovertemplate='<b>Temperature:</b> %{x:.1f}°C<br>' +
                  '<b>Frequency:</b> %{y}<extra></extra>'
))

# Normal distribution curve
mu, sigma = df['Temperature_C'].mean(), df['Temperature_C'].std()
x_norm = np.linspace(df['Temperature_C'].min(), df['Temperature_C'].max(), 100)
y_norm = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5*((x_norm-mu)/sigma)**2)
y_norm = y_norm * (df['Temperature_C'].max() - df['Temperature_C'].min()) * len(df) / 30

fig9.add_trace(go.Scatter(
    x=x_norm,
    y=y_norm,
    mode='lines',
    name='Normal Distribution',
    line=dict(color='red', width=2),
    hovertemplate='<b>Normal Curve</b><br>%{y:.0f}<extra></extra>'
))

# Add vertical lines for mean and median
fig9.add_vline(x=mu, line_width=2, line_dash="dash", line_color="red",
               annotation_text=f'Mean: {mu:.1f}°C', annotation_position="top right")
fig9.add_vline(x=df['Temperature_C'].median(), line_width=2, line_dash="dash", line_color="blue",
               annotation_text=f'Median: {df["Temperature_C"].median():.1f}°C', annotation_position="bottom right")

fig9.update_layout(
    title='🌡️ Temperature Distribution',
    xaxis_title='Temperature (°C)',
    yaxis_title='Frequency',
    template='plotly_white',
    height=500,
    width=800,
    hovermode='x unified'
)

fig9.show()

🌡️ Creating Visualization 9: Temperature Distribution


In [ ]:

# ============================================================================
# VISUALIZATION 10: Interactive Monthly Rainfall by City (Grouped Bar)
# ============================================================================

print("📊 Creating Visualization 10: Monthly Rainfall by City")

# Select top 3 cities
top_cities = df.groupby('City')['Rainfall_mm'].mean().nlargest(3).index
df_top = df[df['City'].isin(top_cities)]
pivot_city_month = df_top.pivot_table(values='Rainfall_mm', index='Month_Name', columns='City', aggfunc='mean')
pivot_city_month = pivot_city_month.reindex(['January', 'February', 'March', 'April', 'May', 'June',
                                              'July', 'August', 'September', 'October', 'November', 'December'])

fig10 = go.Figure()

colors = ['#2E86AB', '#A23B72', '#F18F01']

for i, city in enumerate(top_cities):
    fig10.add_trace(go.Bar(
        x=pivot_city_month.index,
        y=pivot_city_month[city],
        name=city,
        marker_color=colors[i],
        hovertemplate='<b>City:</b> %{customdata}<br>' +
                      '<b>Month:</b> %{x}<br>' +
                      '<b>Rainfall:</b> %{y:.1f} mm<extra></extra>',
        customdata=[city] * len(pivot_city_month)
    ))

fig10.update_layout(
    title='📊 Monthly Rainfall Comparison (Top 3 Cities)',
    xaxis_title='Month',
    yaxis_title='Average Rainfall (mm)',
    template='plotly_white',
    height=500,
    width=800,
    barmode='group',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig10.show()

📊 Creating Visualization 10: Monthly Rainfall by City


In [ ]:
# ============================================================================
# VISUALIZATION 11: Interactive Polar Plot with Hover
# ============================================================================

print("🔄 Creating Visualization 11: Rainfall Seasonality Cycle")

monthly_avg = df.groupby('Month')['Rainfall_mm'].mean().reset_index()
monthly_avg.columns = ['Month', 'Rainfall_mm']

# Add month names for hover
month_names = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_avg['Month_Name'] = monthly_avg['Month'].apply(lambda x: month_names[x-1])

fig11 = go.Figure()

fig11.add_trace(go.Scatterpolar(
    r=monthly_avg['Rainfall_mm'],
    theta=monthly_avg['Month_Name'],
    mode='lines+markers+text',
    name='Rainfall Cycle',
    line=dict(color='#2E86AB', width=3),
    marker=dict(size=12, color='#2E86AB'),
    text=monthly_avg['Rainfall_mm'].round(1),
    textposition='top center',
    hovertemplate='<b>Month:</b> %{theta}<br>' +
                  '<b>Rainfall:</b> %{r:.1f} mm<br>' +
                  '<extra></extra>',
    fill='toself',
    fillcolor='rgba(46, 134, 171, 0.3)'
))

fig11.update_layout(
    title='🔄 Rainfall Seasonality Cycle',
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, monthly_avg['Rainfall_mm'].max() * 1.1]
        ),
        angularaxis=dict(
            direction='clockwise',
            period=12
        )
    ),
    template='plotly_white',
    height=500,
    width=600,
    showlegend=False
)

fig11.show()

🔄 Creating Visualization 11: Rainfall Seasonality Cycle


In [ ]:
# ============================================================================
# PART 4: ANALYSIS AND INSIGHTS GENERATION
# ============================================================================

print("\n" + "="*80)
print("📊 STEP 4: ANALYSIS AND INSIGHTS")
print("="*80)

# 4.1: Basic Statistics
print("\n📈 4.1 BASIC STATISTICS")
print("-"*50)
print(f"Total Records: {len(df):,}")
print(f"Total Rainfall: {df['Rainfall_mm'].sum():,.1f} mm")
print(f"Average Rainfall: {df['Rainfall_mm'].mean():.2f} mm")
print(f"Median Rainfall: {df['Rainfall_mm'].median():.2f} mm")
print(f"Maximum Rainfall: {df['Rainfall_mm'].max():.1f} mm")
print(f"Minimum Rainfall: {df['Rainfall_mm'].min():.1f} mm")
print(f"Standard Deviation: {df['Rainfall_mm'].std():.2f} mm")


📊 STEP 4: ANALYSIS AND INSIGHTS

📈 4.1 BASIC STATISTICS
--------------------------------------------------
Total Records: 2,245
Total Rainfall: 226,204.0 mm
Average Rainfall: 100.76 mm
Median Rainfall: 102.00 mm
Maximum Rainfall: 199.0 mm
Minimum Rainfall: 0.0 mm
Standard Deviation: 58.10 mm


In [ ]:
# 4.2: Monthly Analysis
print("\n📅 4.2 MONTHLY RAINFALL ANALYSIS")
print("-"*50)
monthly_summary = df.groupby('Month_Name')['Rainfall_mm'].agg(['mean', 'sum', 'count']).reset_index()
monthly_summary.columns = ['Month', 'Avg (mm)', 'Total (mm)', 'Days']
monthly_summary = monthly_summary.sort_values('Avg (mm)', ascending=False)
print(monthly_summary.to_string(index=False))



📅 4.2 MONTHLY RAINFALL ANALYSIS
--------------------------------------------------
    Month   Avg (mm)  Total (mm)  Days
 December 111.094118     18886.0   170
 November 106.662651     17706.0   166
      May 102.870270     19031.0   185
September 102.633508     19603.0   191
    April 101.458763     19683.0   194
   August 101.126263     20023.0   198
  January  99.086735     19421.0   196
     June  98.810056     17687.0   179
 February  97.486339     17840.0   183
     July  97.310881     18781.0   193
  October  97.214286     19054.0   196
    March  95.304124     18489.0   194


In [ ]:
# 4.3: Seasonal Analysis
print("\n🌦️ 4.3 SEASONAL RAINFALL ANALYSIS")
print("-"*50)
seasonal_summary = df.groupby('Season')['Rainfall_mm'].agg(['mean', 'sum', 'count']).reset_index()
seasonal_summary.columns = ['Season', 'Avg (mm)', 'Total (mm)', 'Days']
seasonal_summary = seasonal_summary.sort_values('Avg (mm)', ascending=False)
print(seasonal_summary.to_string(index=False))


🌦️ 4.3 SEASONAL RAINFALL ANALYSIS
--------------------------------------------------
 Season   Avg (mm)  Total (mm)  Days
        107.078947      4069.0    38
 Summer 103.907021     54759.0   527
 Autumn 103.249527     54619.0   529
 Winter  98.183953     50172.0   511
Monsoon  97.690391     54902.0   562


In [ ]:
# 4.4: City-wise Analysis
print("\n🏙️ 4.4 CITY-WISE RAINFALL ANALYSIS")
print("-"*50)
city_summary = df.groupby('City')['Rainfall_mm'].agg(['mean', 'sum', 'max', 'min', 'count']).reset_index()
city_summary.columns = ['City', 'Avg (mm)', 'Total (mm)', 'Max (mm)', 'Min (mm)', 'Days']
city_summary = city_summary.sort_values('Avg (mm)', ascending=False)
print(city_summary.to_string(index=False))


🏙️ 4.4 CITY-WISE RAINFALL ANALYSIS
--------------------------------------------------
     City   Avg (mm)  Total (mm)  Max (mm)  Min (mm)  Days
  Chennai 104.418103     48450.0     198.0       1.0   464
Bangalore 102.004717     43250.0     199.0       0.0   424
   Mumbai 100.412148     46290.0     199.0       0.0   461
     Pune 100.065421     42828.0     199.0       0.0   428
    Delhi  96.978632     45386.0     199.0       1.0   468


In [ ]:
# 4.5: Correlation Analysis
print("\n🔗 4.5 CORRELATION ANALYSIS")
print("-"*50)
corr_rainfall_temp = df['Rainfall_mm'].corr(df['Temperature_C'])
corr_rainfall_humidity = df['Rainfall_mm'].corr(df['Humidity_%'])
corr_temp_humidity = df['Temperature_C'].corr(df['Humidity_%'])

print(f"Rainfall vs Temperature: {corr_rainfall_temp:.3f}")
if corr_rainfall_temp > 0:
    print("  → Positive correlation (rainfall increases with temperature)")
else:
    print("  → Negative correlation (rainfall decreases with temperature)")

print(f"\nRainfall vs Humidity: {corr_rainfall_humidity:.3f}")
if corr_rainfall_humidity > 0:
    print("  → Positive correlation (rainfall increases with humidity)")
else:
    print("  → Negative correlation (rainfall decreases with humidity)")

print(f"\nTemperature vs Humidity: {corr_temp_humidity:.3f}")
if corr_temp_humidity > 0:
    print("  → Positive correlation (temperature increases with humidity)")
else:
    print("  → Negative correlation (temperature decreases with humidity)")


🔗 4.5 CORRELATION ANALYSIS
--------------------------------------------------
Rainfall vs Temperature: -0.025
  → Negative correlation (rainfall decreases with temperature)

Rainfall vs Humidity: -0.012
  → Negative correlation (rainfall decreases with humidity)

Temperature vs Humidity: 0.024
  → Positive correlation (temperature increases with humidity)


In [ ]:
# 4.6: Rainfall Category Distribution
print("\n📊 4.6 RAINFALL CATEGORY DISTRIBUTION")
print("-"*50)
category_dist = df['Rainfall_Category'].value_counts()
print(category_dist)
print(f"\nMost common category: {category_dist.index[0]} ({category_dist.values[0]:,} days)")


📊 4.6 RAINFALL CATEGORY DISTRIBUTION
--------------------------------------------------
Rainfall_Category
Extreme     1154
Heavy        534
Moderate     443
Light        114
Name: count, dtype: int64

Most common category: Extreme (1,154 days)


In [ ]:
# 4.7: Rainfall Type Distribution
print("\n🌧️ 4.7 RAINFALL TYPE DISTRIBUTION")
print("-"*50)
type_dist = df['Clean_Rainfall_Type'].value_counts()
print(type_dist)
print(f"\nMost common type: {type_dist.index[0]} ({type_dist.values[0]:,} days)")


🌧️ 4.7 RAINFALL TYPE DISTRIBUTION
--------------------------------------------------
Clean_Rainfall_Type
Light Rain       840
Moderate Rain    524
No Rain          384
Heavy Rain       256
Drizzle          241
Name: count, dtype: int64

Most common type: Light Rain (840 days)


In [ ]:
# 4.8: Find key metrics for insights
rainiest_city = city_summary.iloc[0]['City']
rainiest_avg = city_summary.iloc[0]['Avg (mm)']
driest_city = city_summary.iloc[-1]['City']
driest_avg = city_summary.iloc[-1]['Avg (mm)']
rainiest_season = seasonal_summary.iloc[0]['Season']
rainiest_season_avg = seasonal_summary.iloc[0]['Avg (mm)']
rainiest_month = monthly_summary.iloc[0]['Month']
rainiest_month_avg = monthly_summary.iloc[0]['Avg (mm)']
rainy_days = len(df[df['Rainfall_mm'] > 0])
rainy_pct = (rainy_days / len(df)) * 100
extreme_events = len(df[df['Rainfall_Category'] == 'Extreme'])

In [ ]:
# 4.9: Top Insights
print("\n💡 4.9 KEY INSIGHTS")
print("="*80)

print(f"""
1. 🏆 RAINIEST CITY: {rainiest_city} with {rainiest_avg:.1f} mm average rainfall
   → This city receives the highest rainfall among all cities

2. 🌵 DRIEST CITY: {driest_city} with {driest_avg:.1f} mm average rainfall
   → This city receives the lowest rainfall

3. 🌦️ RAINIEST SEASON: {rainiest_season} with {rainiest_season_avg:.1f} mm average
   → Most rainfall occurs during this season

4. 📅 RAINIEST MONTH: {rainiest_month} with {rainiest_month_avg:.1f} mm average
   → Peak rainfall month

5. ☔ RAINY DAYS: {rainy_days} out of {len(df)} days ({rainy_pct:.1f}%)
   → Percentage of days with rainfall

6. ⚡ EXTREME EVENTS: {extreme_events} days with extreme rainfall (>100mm)
   → Days with very heavy rainfall

7. 📈 CORRELATION INSIGHT:
   → Rainfall has a {abs(corr_rainfall_humidity):.3f} correlation with humidity
   → Rainfall has a {abs(corr_rainfall_temp):.3f} correlation with temperature
""")



💡 4.9 KEY INSIGHTS

1. 🏆 RAINIEST CITY: Chennai with 104.4 mm average rainfall
   → This city receives the highest rainfall among all cities

2. 🌵 DRIEST CITY: Delhi with 97.0 mm average rainfall
   → This city receives the lowest rainfall

3. 🌦️ RAINIEST SEASON:  with 107.1 mm average
   → Most rainfall occurs during this season

4. 📅 RAINIEST MONTH: December with 111.1 mm average
   → Peak rainfall month

5. ☔ RAINY DAYS: 2238 out of 2245 days (99.7%)
   → Percentage of days with rainfall

6. ⚡ EXTREME EVENTS: 1154 days with extreme rainfall (>100mm)
   → Days with very heavy rainfall

7. 📈 CORRELATION INSIGHT: 
   → Rainfall has a 0.012 correlation with humidity
   → Rainfall has a 0.025 correlation with temperature

